In [ ]:
docker-compose -f docker-compose.cloud.yml down -v

docker-compose -f docker-compose.cloud.yml up -d

In [ ]:
python scripts/setup_minio.py 


: 

In [ ]:
pip freeze > requirements.txt
pip install -r requirements.txt


alembic init alembic

alembic revision --autogenerate -m "add_event_fields_and_fcm_token" 

Bản thiết kế lược đồ PostgreSQL (db_schema_v1.sql) của bạn hiện tại đã rất hoàn thiện và bám sát hoàn toàn vào kiến trúc hệ thống cũng như kế hoạch 8 tuần trong tài liệu
.
Dưới đây là phân tích chi tiết vì sao bản thiết kế này đã đạt yêu cầu và một vài điểm cộng kỹ thuật bạn đã thực hiện đúng:
1. Phù hợp hoàn hảo với vai trò các lớp (Layers)
Hỗ trợ Lớp 1 (Edge): Bảng cameras với rtsp_url là thông tin đầu vào thiết yếu cho pipeline DeepStream
. Bảng events đã có trường person_id để lưu ID từ nvtracker (ByteTrack), giúp theo dõi đối tượng xuyên suốt các frame hình
.
Hỗ trợ Lớp 2 (Cloud):
Bảng users phục vụ trực tiếp cho nhiệm vụ Auth API (JWT login/register) ở Tuần 2
.
Trường vlm_verdict trong bảng alerts sẵn sàng cho việc lưu kết quả xác minh từ Gemini/GPT-4V ở Tuần 4
.
Bảng confidence_log đáp ứng đúng yêu cầu của MLOps collector để lọc dữ liệu phục vụ việc tái huấn luyện mô hình (retrain trigger)
.
Hỗ trợ Lớp 3 (App): Các Index bạn tạo (idx_events_camera_timestamp) sẽ giúp màn hình History trên ứng dụng Flutter truy vấn dữ liệu cực nhanh ngay cả khi số lượng sự kiện lên tới hàng nghìn
.
2. Các điểm sáng trong thiết kế của bạn
Lưu trữ Multimedia: Việc tách biệt image_url và video_clip_url trong bảng events rất chính xác. Điều này phù hợp với kế hoạch Tuần 6 là sử dụng MinIO để lưu trữ bằng chứng hình ảnh và các clip 10 giây để người dùng xem lại trên App
.
Tính toàn vẹn dữ liệu: Sử dụng ON DELETE CASCADE cho các khóa ngoại (camera_id, event_id) là lựa chọn thông minh. Khi bạn xóa một camera (nhiệm vụ Camera CRUD ở Tuần 2), toàn bộ sự kiện và cảnh báo liên quan sẽ được dọn dẹp sạch sẽ, tránh dữ liệu rác
.
Theo dõi trạng thái thiết bị: Trường last_heartbeat trong bảng cameras sẽ là nơi tiếp nhận dữ liệu Telemetry REST từ Edge gửi về mỗi phút, giúp Cloud biết được camera nào đang offline để báo cáo lên hệ thống
.
3. Lưu ý nhỏ để hoàn thiện hơn (Optional)
Định dạng BBox: Trong bảng events, bạn dùng JSONB cho bbox_json. Đây là lựa chọn tốt nhất. Tuy nhiên, khi nhận API Contract từ Dũng vào ngày thứ 3 của Tuần 1, bạn cần thống nhất xem Edge sẽ gửi tọa độ dạng [x, y, w, h] hay [x1, y1, x2, y2] để xử lý đồng nhất
.
Class Labels: Cột class_label nên khớp với 4 lớp mà Duy sẽ train: đứng, ngồi, nằm, other (hoặc fall khi đã qua bộ lọc 2 giây)
.
Kết luận: Bản thiết kế này đã đạt chuẩn (Ready) để bạn thực hiện bước tiếp theo là xây dựng Docker Compose local và bắt đầu viết các FastAPI routers cho Tuần 2
. Bạn có thể tự tin sử dụng file này để bàn giao cho cột mốc (milestone) của Tuần 1.

In [ ]:
#Rabbitqm
http://localhost:15672

#activate virtual environment



.\.venv\Scripts\Activate.ps1

cd fall_detection_cloud

docker-compose -f docker-compose.cloud.yml up -d

In [ ]:
# BÀN GIAO TUẦN 6 - DEPLOYMENT & INTEGRATION

## 1. Triển khai và Ổn định hệ thống (Deployment & Stability)

### Cloud Stack Deployment
**File bàn giao:** `docker-compose.cloud.yml`
**Nội dung:**
- FastAPI Backend (port 8000)
- PostgreSQL Database (port 5432)
- RabbitMQ + MQTT Plugin (ports 5672, 1883, 15672)
- MinIO Object Storage (ports 9000, 9001)
- MediaMTX Live Streaming (ports 8554, 8888, 8889, 1935)

**Cách triển khai:**
```bash
# Trên VPS hoặc server phòng Lab
docker-compose -f docker-compose.cloud.yml up -d
```

### Auto-Recovery Configuration
**File bàn giao:** `docker-compose.cloud.yml`
**Cấu hình restart policy:**
- Tất cả services có `restart: always`
- Tự động khởi động khi server reboot
- Tự động restart khi container crash

### Security & Backup
**File bàn giao:** `scripts/backup_database.sh` (cần tạo)
**Nội dung backup script:**
```bash
#!/bin/bash
# Backup PostgreSQL database
BACKUP_DIR="/backups/postgres"
DATE=$(date +%Y%m%d_%H%M%S)
docker exec fall_db pg_dump -U postgres falldetection > $BACKUP_DIR/falldetection_$DATE.sql
# Keep last 7 days
find $BACKUP_DIR -name "falldetection_*.sql" -mtime +7 -delete
```

---

## 2. Phát triển các API mở rộng

### Telemetry API
**File bàn giao:** `app/api/v1/endpoints/telemetry.py` (cần tạo)
**Endpoints:**
- `POST /telemetry` - Nhận telemetry từ Edge
- `GET /telemetry/cameras/{cam_id}` - Lấy telemetry logs theo camera
- `GET /telemetry/cameras/{cam_id}/latest` - Lấy telemetry mới nhất

**Schema bàn giao:** `app/schemas/telemetry.py`
**Trường dữ liệu:**
- `fps_pgie`, `fps_sgie`: FPS detectors
- `active_tracks`: Số track đang active
- `ram_used_mb`, `ram_total_mb`: RAM usage
- `cpu_temp_c`, `gpu_temp_c`: Nhiệt độ
- `cpu_usage_pct`: CPU usage
- `disk_free_gb`: Disk space
- `mqtt_connected`: Trạng thái MQTT
- `last_event_sent_utc`: Timestamp event gần nhất

**Bàn giao cho Dũng (Edge):**
- Endpoint URL: `http://<VPS_IP>:8000/api/telemetry`
- Payload format: Xem `mqtt_schema_v2.json`
- Frequency: Gửi mỗi 1 phút

### Admin Rules API
**File bàn giao:** `app/api/v1/endpoints/rules.py` (cần tạo)
**Endpoints:**
- `GET /rules/cameras/{cam_id}` - Lấy cấu hình rule theo camera
- `PUT /rules/cameras/{cam_id}` - Cập nhật cấu hình rule
- `GET /rules/cameras` - Lấy tất cả rules

**Schema bàn giao:** `app/schemas/rule.py`
**Trường cấu hình:**
- `time_window_sec`: Cửa sổ thời gian (mặc định: 2s)
- `min_lying_frames`: Số frame tối thiểu (mặc định: 5)
- `min_confidence_sgie`: Ngưỡng confidence SGIE (mặc định: 0.6)
- `enable_vlm_verify`: Bật/tắt VLM verification
- `is_active`: Bật/tắt rule

**Bàn giao cho Dũng (Edge):**
- Edge đọc cấu hình mỗi 5 phút
- Cache locally

**Bàn giao cho Duy (App):**
- Admin interface để điều chỉnh cấu hình

---

## 3. Kết nối thực tế và Kiểm thử cuối

### Kết nối với Jetson thật
**File bàn giao:** `docs/HANDOVER_EDGE.md` (đã tạo)
**Kiểm tra kết nối:**
```bash
# Test MQTT connection
mosquitto_pub -h <VPS_IP> -p 1883 -u <user> -P <pass> \
  -t "events/cam_01/fall" -m '{"test": "connection"}'

# Test RTSP push
ffmpeg -re -i /dev/video0 -c:v libx264 -preset ultrafast \
  -f rtsp rtsp://<VPS_IP>:8554/cam_01
```

### Full Trigger Test
**Kịch bản test:**
1. Người ngã trước camera (Jetson)
2. Edge phát hiện fall (DeepStream)
3. Edge gửi MQTT event lên Cloud
4. Cloud validate → dedup → VLM verify
5. Cloud tạo alert → gửi FCM → gửi Telegram
6. App nhận FCM notification
7. App xem live stream (HLS)
8. App xem clip (MinIO presigned URL)

**File test:** `scripts/test_full_trigger.py` (cần tạo)

### Load Test
**File bàn giao:** `scripts/test_load.py` (cần tạo)
**Kịch bản test:**
- Giả lập 50 events dồn dập gửi cùng lúc
- Kiểm tra asyncio.Queue không overflow
- Kiểm tra không có event bị mất
- Kiểm tra VLM latency vẫn < 3s

---

## 4. Definition of Done - Tuần 6

### Checklist hoàn thành:
- [ ] Cloud stack deployed lên VPS/server phòng Lab
- [ ] Tất cả Docker container có restart policy
- [ ] Database backup script hoạt động (cron job)
- [ ] Telemetry API hoạt động, Edge gửi được telemetry
- [ ] Admin Rules API hoạt động, Edge đọc được cấu hình
- [ ] Kết nối MQTT với Jetson thật thành công
- [ ] Kết nối RTSP với Jetson thật thành công
- [ ] Full trigger test pass (ngã → FCM → App)
- [ ] Load test 50 events pass (không mất event)
- [ ] Bàn giao Telemetry API cho Dũng
- [ ] Bàn giao Admin Rules API cho Dũng
- [ ] Bàn giao Admin Rules API cho Duy

### Files cần bàn giao:
1. `docker-compose.cloud.yml` - Cloud stack
2. `.env.example` - Environment variables template
3. `scripts/backup_database.sh` - Backup script
4. `app/api/v1/endpoints/telemetry.py` - Telemetry API
5. `app/api/v1/endpoints/rules.py` - Admin Rules API
6. `app/schemas/telemetry.py` - Telemetry schema
7. `app/schemas/rule.py` - Rule schema
8. `scripts/test_full_trigger.py` - Full trigger test
9. `scripts/test_load.py` - Load test
10. `docs/HANDOVER_EDGE.md` - Handover cho Dũng (đã có)
11. `docs/HANDOVER_APP.md` - Handover cho Duy (đã có)
12. `docs/DEPLOYMENT.md` - Deployment guide (cần tạo)

1. Các thành phần bạn đã liệt kê:
RUN apt-get update && apt-get install -y libpq-dev gcc: Đây là bước quan trọng để cài đặt các thư viện hệ thống cần thiết cho psycopg2
. Nếu thiếu dòng này, container sẽ không thể kết nối với PostgreSQL để thực hiện nhiệm vụ khởi tạo lược đồ dữ liệu (db_schema_v1.sql)
.
COPY requirements.txt . & RUN pip install...: Dòng này thực hiện cài đặt các "vũ khí" bạn cần cho dự án như FastAPI, SQLAlchemy, Alembic, và RabbitMQ (Pika/Aio-pika)
.
COPY . .: Sao chép toàn bộ cấu trúc dự án (routers, schemas, models) vào bên trong container để sẵn sàng vận hành
.
CMD [...]: Lệnh khởi chạy server Uvicorn. Lưu ý tham số --reload cực kỳ hữu ích trong Tuần 1 và Tuần 2 khi bạn đang phát triển các tính năng như Auth API và Camera CRUD, giúp container tự động cập nhật ngay khi bạn sửa code
.